# Figure 1 — song and preference fields

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports and input recording

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import wavfile
import scipy
from scipy.signal import spectrogram, hilbert

DATA_DIR = ROOT / 'data/song'
STEM = "SI2203"

## Load the audio

In [ ]:
sample_rate, audio = wavfile.read(DATA_DIR / "BimSong_excerpt.wav")
audio = audio[:400_000]
duration = len(audio) / sample_rate
time = np.arange(len(audio)) / sample_rate

## Find pulses and chirps

In [ ]:
sng = audio[:int(duration*sample_rate)]
plt.figure(figsize=(20, 4))
plt.subplot(211)
plt.plot(sng)
env = np.convolve(np.abs(hilbert(sng)), np.ones(800)/800)
plt.plot(env)

pulse_train = (np.diff(env)>2).astype(float)
pulse_train = scipy.signal.medfilt(pulse_train, 201)
plt.subplot(212)
plt.plot(pulse_train)

pulse_onsets = np.where(np.diff(pulse_train) > 0)[0]
pulse_offsets = np.where(np.diff(pulse_train) < 0)[0]

phonemes = []
for on, off in zip(pulse_onsets, pulse_offsets):
    phonemes.append((on, off, 'P'))

words = []
for on, off in zip(pulse_onsets[::4], pulse_offsets[3::4]):
    words.append((on, off, 'Chirp'))

## Calculate the spectrogram

In [ ]:
frequencies, spec_time, power = spectrogram(
    audio, fs=sample_rate, window="hann", nperseg=1024, noverlap=780, scaling="spectrum"
)
power_db = 10 * np.log10(power + np.finfo(float).eps)

## Plot the annotated song

In [ ]:
fig, (ax_wave, ax_spec, ax_phn, ax_word) = plt.subplots(
    4, 1, figsize=(10, 5), sharex=True, constrained_layout=True,
    gridspec_kw={"height_ratios": (1, 2, 0.1, 0.1)}
)
fig.suptitle('Gryllus bimaculatus song', fontsize=15)

ax_wave.plot(time, audio, color="black", linewidth=0.5)
ax_wave.set(ylabel="Amplitude", xlim=(0, duration))
ax_wave.axis('off')

mesh = ax_spec.pcolormesh(spec_time, frequencies / 1000, power_db, shading="auto", cmap="Greys", rasterized=True)
mesh.set(clim=10)
ax_spec.set(ylabel="Frequency (kHz)", ylim=(0, sample_rate / 2000 / 4))
fig.colorbar(mesh, ax=ax_spec, pad=0.01, label="Power (dB)")

for ax, segments, rotation in ((ax_phn, phonemes, 0), (ax_word, words, 0)):
    colors = {'P': '#9b7fe0', 'Chirp': '#f36b60'}
    for start, end, label in segments:
        start, end = start / sample_rate, end / sample_rate
        ax.axvspan(start, end, color=colors[label], alpha=0.25)
        ax.text((start + end) / 2, 0.5, label, ha="center", va="center",
                rotation=rotation, fontsize=8)
    ax.set(ylim=(0, 1), yticks=[])
    ax.axis('off')

ax_phn.set_ylabel("Phonemes")
ax_word.set(ylabel="Words", xlabel="Time (s)")
fig.savefig(f'{FIGURES}/fig_1A-D.pdf', bbox_inches='tight')

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from Gbim_import_dat import get_Gbim_data_chirpfield, get_Gbim_data_DC

from metpy.interpolate import interpolate_to_points
import pandas as pd

In [ ]:
MAX = 40
step = 10
cpauMax = 1000 + MAX
max_len = cpauMax

## Load experimental responses

In [ ]:
[CDUR, CPAU, CPER, DCs, NPUL, rXY]= get_Gbim_data_chirpfield()
[CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]=get_Gbim_data_DC()

keys = ['CDUR', 'CPAU', 'CPER', 'CDC', 'NPUL', 'rXY']
data_cpf = dict(zip(keys, [CDUR, CPAU, CPER, DCs, NPUL, rXY]))
data_dc = dict(zip(keys, [CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]))

data_all = {key: np.concatenate((data_cpf[key], data_dc[key])) for key in keys}

## Interpolate the chirp field

In [ ]:
duri0 = paui0 = np.arange(0, max_len, step)
duri, paui = np.meshgrid(duri0,paui0)

CDURinterp=np.concatenate((CDUR,[0, 0]))
CPAUinterp=np.concatenate((CPAU,[0, 1000]))
rXYinterp=np.concatenate((rXY,[0, 0]))

points = np.array([CDURinterp + np.random.randn(*CDURinterp.shape)/1_000, CPAUinterp+ np.random.randn(*CPAUinterp.shape)/1_000]).T
new_points = np.array([duri, paui]).T.reshape((-1, 2))

cpf_behave = interpolate_to_points(points, np.clip(rXYinterp, 0, np.inf), new_points, interp_type='natural_neighbor', minimum_neighbors=4, rbf_smooth=0)
cpf_behave = np.maximum(cpf_behave, 0)
cpf_behave = cpf_behave.reshape((len(duri), len(paui))).T

## Load the pulse-response table

In [ ]:
df = pd.read_csv(ROOT / 'data/song/BIM_ppf_from_old_model_paper.csv')
df

## Interpolate the pulse field

In [ ]:
MAX = 40
step = 1
ppauMax = 100 + MAX
max_len = ppauMax

duri0 = paui0 = np.arange(0, max_len, step)
pduri, ppaui = np.meshgrid(duri0,paui0)

PDURinterp=np.concatenate((df['pdur'],[0, 0, 0]))
PPAUinterp=np.concatenate((df['ppau'],[0, 40, 80]))
rXYinterp=np.concatenate((df['resp'],[0, 0, 0]))

points = np.array([PDURinterp + np.random.randn(*PDURinterp.shape)/1_000, PPAUinterp+ np.random.randn(*PPAUinterp.shape)/1_000]).T
new_points = np.array([pduri, ppaui]).T.reshape((-1, 2))

ppf_behave = interpolate_to_points(points, np.clip(rXYinterp, 0, np.inf), new_points, interp_type='natural_neighbor', minimum_neighbors=4, rbf_smooth=0)
ppf_behave = np.maximum(ppf_behave, 0)
ppf_behave = ppf_behave.reshape((len(pduri), len(ppaui))).T

## Plot preference fields

In [ ]:
plt.figure(figsize=(9, 3))
plt.subplot(131)
plt.pcolormesh(pduri, ppaui, ppf_behave / np.nanmax(ppf_behave), cmap='Purples', rasterized=True)
plt.plot(df['pdur'], df['ppau'], '.k')
plt.axis('square')
plt.xlim(0, 80)
plt.ylim(0, 80)
plt.xlabel('Pulse duration [ms]')
plt.ylabel('Pulse pause [ms]')
plt.xticks([0, 80])
plt.yticks([0, 80])
plt.clim(0, 1)
cb = plt.colorbar(shrink=0.4, label='Phonotaxis')
cb.set_ticks([0, 1])

plt.subplot(132)
plt.pcolormesh(duri/1000, paui/1000, cpf_behave / np.nanmax(cpf_behave), cmap='Reds', rasterized=True)
plt.plot(CDUR/1000, CPAU/1000, '.k')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel('Chirp duration [s]')
plt.ylabel('Chirp pause [s]')
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.clim(0, 1)
cb = plt.colorbar(shrink=0.4, label='Phonotaxis')
cb.set_ticks([0, 1])


plt.tight_layout()
plt.savefig(f'{FIGURES}/fig_1E-G.pdf', bbox_inches='tight')